# Unit Testing for Data Science 

## Exploratory Data Analysis

This notebook was created for the [Berlin WiMLDS Workshop](https://github.com/wimlds/berlin-tdd-workshop) and it contains exploratory data analysis of a dataset based on the [Pockets dataset from The Pudding](https://github.com/the-pudding/pockets). In order to try out different imputation methods we removed 10% of the price values.

You can use this notebook to get an overview of the dataset or write functions for the exercises, before transfering them to the appropriate Python scripts. You can find the python scripts in the ```src.data_pipeline``` directory!

<pre>
ds-unit-testing
├──src/
│  ├──data_pipeline/
│  │  ├── __init__.py
│  │  ├── imputation.py
│  │  └── transformation.py
</pre>

So let's get started. First we have to import the libraries we want to use. For this notebook we only need pandas and numpy.

In [ ]:
# because ipytest throws quite a few warnings we will just ignore them
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

# For pandas we can set options like rounding floats to 2 decimals
pd.set_option('display.float_format', lambda x: '%.2f' % x)

Now let's load the data into a dataframe:

In [ ]:
df = pd.read_csv(
    "data/measurementRectanglesMissing.csv"
) 

Now we can have a look how the data looks like.

In [ ]:
# Let's have a look at the 5 first rows
df.head()

## Missing values

Now let's have a look if we have missing values in the data frame.

In [ ]:
# how many missing values in 'price' column?
print("Number of missing values: ", df.price.isnull().sum())

Now, dealing with missing values is an important step in our workflow, we will only touch it shortly in this notebook, but we will have an extra day for that topic. So for now let's say we want to impute all missing values with the mean of the regarding feature. The ```->``` in functions define what you expect the function to return.

In [ ]:
# fucntion to impute NaNs with mean value
def impute(series: pd.Series) -> pd.Series:
    mean = series.mean()
    return series.fillna(mean)

In [ ]:
# Let's first make a copy of our dataframe (if we want to work later on the original data or want to test different imputations)
df_impute = df.copy()
# And now let's impute the missing values in our price column
df_impute["price"] = impute(df_impute["price"])

In [ ]:
# And check again if the missings are gone
df_impute.price.isnull().sum() 

### Test the imputation

In the notebook before we learned how important it is to test your code, but pandas also offers the possibility to test your data. With ```pandas.testing``` you can compare Pandas DataFrames and Pandas Series. So let's test if our imputation function leeds to the expected outcome.

In [ ]:
import pytest
import ipytest
from pandas.testing import assert_series_equal
from pandas.testing import assert_frame_equal

ipytest.autoconfig()

In [ ]:
%%ipytest -qq


@pytest.mark.parametrize("input_series, expected_result", [
    (pd.Series([1.0, np.nan, 3.0]),  pd.Series([1.0, 2.0, 3.0])),
    (pd.Series([1, 2, 3, np.nan, 4, np.nan]), pd.Series([1, 2, 3, 2.5, 4, 2.5]))
])
def test_impute(input_series, expected_result):
    assert_series_equal(impute(input_series), expected_result)

#### Exercise 1
Add functions to impute missing values with min/max values, then add unit tests.

Don't forget to add your functions to ```imputation.py``` in the src.data_pipeline directory and your unit tests to test_imputation.py

In [ ]:
# write your functions/unit tests here
# don't forget to transfer to appropriate scripts after


### Explore the dataset

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df["brand"].nunique()

In [ ]:
df["style"].nunique()

In [ ]:
df["menWomen"].nunique()

We can see that we have some features that we can transform to categorical features.

In [ ]:
#transforming categorical columns to categorical type: brand style menWomen
df["brand"] = df["brand"].astype("category")
df["style"] = df["style"].astype("category")
df["menWomen"] = df["menWomen"].astype("category")

#### Are womens jeans more expensive?

In [ ]:
df.price.describe()

In [ ]:
# bin prices by min/max value, upper/lower percentiles
bins = [9, 50, 74, 96, 250]
df['price_binned'] = pd.cut(df['price'], bins)

In [ ]:
# and look at the comparision between the prices for men and women.
pd.crosstab(df["menWomen"], df.price_binned)

Women jeans are not really more expensive.

#### Are women pockets smaller?

In [ ]:
df.pocketArea.describe()

In [ ]:
# bin pocketArea by min/max value, upper/lower percentiles
bins = [4454, 5906, 8619, 10725, 13103]
df['pocket_binned'] = pd.cut(df['pocketArea'], bins)

In [ ]:
# and look at the comparision between the prices for men and women. 
pd.crosstab(df["menWomen"], df.pocket_binned, normalize="columns")

And here we see that women's pocket sizes are smaller.

### Data Transformations 

Also Feature Engineering is an important step during our workflow. So maybe we could create a women pocket size score for each brand.

To get this score we let's first create a new column ```is_size_greater_than_average```

Add column is_size_greater_than_average (per brand / per dataset) -> with values from 0 to 2, the number of women jeans that are larger than average in that brand

In [ ]:
# Creating a column with data transformation of another column
# This function is also in scripts/transformation.py
def is_greater_than_average(series: pd.Series) -> pd.Series :
    avg = series.mean()
    new_series = [0 if x <= avg else 1 for x in series]

    return pd.Series(new_series)

In [ ]:
# with the function above we now can create a new column in our data frame
df["size_greater_than_average"] = is_greater_than_average(df["pocketArea"])
#and calculate in how many rows we have pockets greater than average.
print("How many pockets are of size greater than average?: ", df["size_greater_than_average"].sum())

And again we can write tests for this function and test if it outputs what we expect. In this case we can also write tests that we expect to fail, normally it is not the best practice to do that, but there is the possibility and we included it so that you have seen it.

In [ ]:
%%ipytest -s

@pytest.mark.parametrize("input_series, expected_result", [
    (pd.Series([1, 2, 3, 2.5, 4]), pd.Series([0, 0, 1, 0, 1])),
    (pd.Series([]), pd.Series([])),
    (pd.Series([10, 10, 10, 10]), pd.Series([0, 0, 0, 0]))
])
def test_is_greater_than_average(input_series, expected_result):
    output_series = is_greater_than_average(series=input_series)
    assert_series_equal(output_series, expected_result)
    assert  type(output_series) is pd.Series

# you can skip tests if you don't want to run
@pytest.mark.skip(reason="it fails")
def test_is_greater_than_average_with_nan():
    input_series = pd.Series([1,2,np.nan])
    expected_series = pd.Series([0,1,np.nan]) # or exception
    output_series = is_greater_than_average(series=input_series)
    assert_series_equal(output_series, expected_series)
    
#or you can use the xfail marker to indicate that you expect a test to fail
@pytest.mark.xfail
def test_is_greater_than_average_with_nan():
    input_series = pd.Series([1,2,np.nan])
    expected_series = pd.Series([0,1,np.nan]) # or exception
    output_series = is_greater_than_average(series=input_series)
    assert_series_equal(output_series, expected_series)

To create the score by brand and gender we will first group our data freame by brand and gender and than sum up the "size_greater_than_average" column.

In [ ]:
# group our data freame by brand and gender and than sum up the "size_greater_than_average" column
aggr = df.groupby(by=["brand", "menWomen"],as_index=False)["size_greater_than_average"].sum()

aggr

Now let's put that into a function.

In [ ]:
def get_sum_score_by_brand_and_gender(
    frame: pd.DataFrame, 
    brand_col="brand", 
    gender_col="menWomen", 
    score_by="size_greater_than_average") -> pd.DataFrame :
    
    aggr = frame.groupby(by=[brand_col, gender_col],as_index=False)[score_by].sum()
    return aggr

In [ ]:
aggr = get_sum_score_by_brand_and_gender(df, "brand", "menWomen", "size_greater_than_average")
aggr

And of course test the function again. Now we compare two data frames!

In [ ]:
%%ipytest -s

@pytest.mark.parametrize("input_frame, expected_result", [
    (pd.DataFrame({'brand': ['Abercrombie', 'Abercrombie', 'Abercrombie', 'Abercrombie'], 
              'menWomen': ['men', 'men', 'women', 'women'], 
              'size_greater_than_average': [1,1,0,1]}), 
     pd.DataFrame({'brand': ['Abercrombie', 'Abercrombie'], 
              'menWomen': ['men', 'women'], 
              'size_greater_than_average': [2,1]})),
    (pd.DataFrame({'brand': ['Abercrombie', 'Calvin Klein', 'Abercrombie', 'Abercrombie'], 
              'menWomen': ['men', 'men', 'women', 'women'], 
              'size_greater_than_average': [1,1,0,0]}), 
     pd.DataFrame({'brand': ['Abercrombie', 'Abercrombie','Calvin Klein'], 
              'menWomen': ['men', 'women','men'], 
              'size_greater_than_average': [1,0,1]}))
    
])
def test_is_greater_than_average(input_frame, expected_result):
    output_frame = get_sum_score_by_brand_and_gender(input_frame, "brand", "menWomen", "size_greater_than_average")
    assert_frame_equal(output_frame, expected_result)
    assert  type(output_frame) is pd.DataFrame

And the last step is to check if there is a brand where the score is the highest for women!

In [ ]:
aggr[aggr.menWomen == "women"].sort_values(by='size_greater_than_average',ascending=False)

So only for Abercrombie that is the case!

Have a look into the ```src/data_pipeline``` directory and in the ```tests``` directory you will find all tests and examples also in python files. If you want to run the tests in your terminal simply use:

```zsh
$ python -m pytest
```

to run all tests at once or:

```zsh 
$ python -m pytest -q tests/<name of the test>
```

For more output you can also use ```-s``` instead of ```-q```.